In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as colors

import numpy as np

import cartopy.crs as ccrs

import torch
import torch.nn as nn
import torch.utils.data as data
import torch_geometric
from torch.nn import Sequential as Seq, Linear, ReLU
from matplotlib.animation import FuncAnimation
import torch.distributed as dist

import sys

sys.path.append("../src/")
from models.vit import *
from utils.data_utils import *
from utils.climate_utils import *
from utils.subgrid_utils import *
from utils.train_utils import *
from utils.eval_utils import *

from torch.utils.data import Dataset, DataLoader
import os
import sys
import random
import copy

In [2]:
exp_num_in = "3"
exp_num_extra = "12"
exp_num_out = "2"


mse = torch.nn.MSELoss()

region = "Gulf_Stream_Ext"
network = "U_net"


interval = 1

N_samples = 4000
N_val = 200
# N_samples = 750
# N_val = 100
N_test = 1000

factor = 10

hist = 0

lag = 1

steps = 8

Nb = 4

if len(sys.argv) > 4:
    n_cond = int((len(sys.argv) - 4) / 2)

str_video = ""

try:
    for i in range(n_cond):
        if type(globals()[sys.argv[int(4 + i * 2)]]) == str:
            temp = str(sys.argv[int(5 + i * 2)])
            exec(sys.argv[int(4 + i * 2)] + "= temp")
            if sys.argv[int(4 + i * 2)] == "network":
                continue
            str_video += "_" + sys.argv[int(4 + i * 2)] + "_" + sys.argv[int(5 + i * 2)]
        elif type(globals()[sys.argv[int(4 + i * 2)]]) == int:
            exec(
                sys.argv[int(4 + i * 2)] + "=" + "int(" + sys.argv[int(5 + i * 2)] + ")"
            )
            str_video += "_" + sys.argv[int(4 + i * 2)] + "_" + sys.argv[int(5 + i * 2)]
    print(str_video)
except:
    print("no cond")


if region == "Kuroshio":
    lat = [15, 41]
    lon = [-215, -185]
elif region == "Kuroshio_Ext":
    lat = [5, 50]
    lon = [-250, -175]
elif region == "Gulf_Stream":
    lat = [25, 50]
    lon = [-70, -35]
elif region == "Gulf_Stream_Ext":
    lat = [27, 50]
    lon = [-82, -35]
elif region == "Tropics":
    lat = [-5, 25]
    lon = [-95, -65]
elif region == "Tropics_Ext":
    lat = [-5, 25]
    lon = [-115, -45]
elif region == "South_America":
    lat = [-60, -30]
    lon = [-70, -35]
elif region == "Africa":
    lat = [-50, -20]
    lon = [5, 45]
elif region == "Quiescent":
    lat = [-42.5, -17.5]
    lon = [-155, -120]
elif region == "Quiescent_Ext":
    lat = [-55, -10]
    lon = [-170, -110]
elif region == "Pacific":
    lat = [-35, 35]
    lon = [-230, -80]
elif region == "Indian":
    lat = [-30, 28]
    lon = [30, 79]
elif region == "Africa_Ext":
    lat = [-55, -15]
    lon = [-5, 55]

s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val


device = "cuda"


inpt_dict = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "ur", "vr"],
    "3": ["um", "vm", "Tm"],
    "4": ["um", "vm", "ur", "vr", "Tm", "Tr"],
    "5": ["ur", "vr"],
    "6": ["ur", "vr", "Tr"],
    "7": ["Tm"],
    "8": ["Tm", "Tr"],
    "9": ["u", "v"],
    "10": ["u", "v", "T"],
    "11": ["tau_u", "tau_v"],
    "12": ["tau_u", "tau_v", "t_ref"],
}
extra_dict = {
    "1": ["ur", "vr"],
    "2": ["ur", "vr", "Tm"],
    "3": ["Tm"],
    "4": ["ur", "vr", "Tm", "Tr"],
    "5": [],
    "6": ["um", "vm"],
    "7": ["um", "vm", "Tm"],
    "8": ["um", "vm", "Tm", "Tr"],
    "9": ["ur", "vr", "tau_u", "tau_v"],
    "10": ["tau_u", "tau_v"],
    "11": ["t_ref"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "13": ["ur", "vr", "Tr", "tau_u", "tau_v", "t_ref"],
}
out_dict = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "Tm"],
    "3": ["ur", "vr"],
    "4": ["ur", "vr", "Tr"],
    "5": ["u", "v"],
    "6": ["u", "v", "T"],
}

grids = xr.open_dataset("/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc")

if region == "global_25":
    grids = xr.open_dataset("/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc")

elif "global" in region:
    grids = coarse_grid(grids, factor)

else:
    grids = grids.sel(
        {"yu_ocean": slice(lat[0], lat[1]), "xu_ocean": slice(lon[0], lon[1])}
    )


area = torch.from_numpy(grids["area_C"].to_numpy()).to(device=device)

inputs = inpt_dict[exp_num_in]
extra_in = extra_dict[exp_num_extra]
outputs = out_dict[exp_num_out]

str_in = "".join([i + "_" for i in inputs])
str_ext = "".join([i + "_" for i in extra_in])
str_out = "".join([i + "_" for i in outputs])

print("inputs: " + str_in)
print("extra inputs: " + str_ext)
print("outputs: " + str_out)

N_atm = len(extra_in)
N_in = len(inputs)
N_extra = N_atm + N_in
N_out = len(outputs)

num_in = int((hist + 1) * N_in + N_extra)

no cond
inputs: um_vm_Tm_
extra inputs: tau_u_tau_v_t_ref_
outputs: um_vm_Tm_


In [3]:
# inputs, extra_in, outputs = gen_data_025_lateral(inputs,extra_in,outputs,lag,lat,lon,Nb)

# wet = xr.zeros_like(inputs[0][0])
# # inputs[0][0,12,12] = np.nan
# for data in inputs:
#     wet +=np.isnan(data[0])
# wet = np.isnan(xr.where(wet==0,np.nan,0))
# wet = np.nan_to_num(wet.to_numpy())
# wet = torch.from_numpy(wet).type(torch.float32).to(device="cpu")
# wet_bool = np.array(wet.cpu()).astype(bool)
# data_in_test = gen_data_in_test(0,e_test,N_test,lag,hist,inputs,extra_in)
# data_out_test = gen_data_out_test(0,e_test,N_test,lag,hist,outputs)
# test_data = data_CNN_Lateral(data_in_test,data_out_test,wet.to(device = "cpu"),N_atm,Nb,device=device)
# torch.save(test_data, '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/test_data_{0}.pt'.format('steps_'+str(steps)+'_'+region+"_in_"+str_in+"ext_"+str_ext+"N_samples_"+str(N_samples)))

test_data = torch.load(
    "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/test_data_{0}.pt".format(
        "steps_"
        + str(steps)
        + "_"
        + region
        + "_in_"
        + str_in
        + "ext_"
        + str_ext
        + "N_samples_"
        + str(N_samples)
    )
)


mean_out = test_data.norm_vals["m_out"]
std_out = test_data.norm_vals["s_out"]

In [4]:
model = VisionTransformer(
    in_channels=num_in,
    patch_size=16,
    emb_size=768,
    num_layers=6,
    num_heads=8,
    output_channels=N_in,
    img_size=[*test_data[0][0].shape[1:]],
)

model = model.to(device)

model = nn.DataParallel(model)  # if unexpected key with module


saved_nets_path = "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-03-07-1GPU_train_CosineAnnealWarmLR_T10/saved_nets/"
short_model_name = "ViT_best_steps_steps_8_"
full_model_name = (
    short_model_name
    + region
    + "_Test_in_"
    + str_in
    + "ext_"
    + str_ext
    + "_out"
    + str_in
    + "N_train_"
    + str(N_samples)
    + "_Lateral_Data_025_no_smooth"
)
model.load_state_dict(
    torch.load(
        saved_nets_path + full_model_name + ".pt", map_location=torch.device(device)
    )
)

/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


<All keys matched successfully>

In [5]:
pred_model_path = (
    "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/" + full_model_name
)
if not os.path.isdir(pred_model_path):
    os.makedirs(pred_model_path)

In [6]:
N_run = 5
len_run = 200

for ns in [4000]:
    for rand_ind in range(1, 4):
        data_shape = test_data[0][1].shape
        model_pred = np.zeros(
            (int(N_run * len_run), data_shape[1], data_shape[2], data_shape[0])
        )

        for i in range(N_run):
            print(ns, rand_ind)
            temp = copy.deepcopy(test_data)
            temp.input = temp.input[int(i * len_run) : int((i + 1) * len_run)]
            temp.output = temp.output[int(i * len_run) : int((i + 1) * len_run)]
            temp.size = len_run

            model_pred_temp = recur_pred_lateral(
                len_run, temp, model, hist, N_in, N_extra, Nb
            )
            print("data_gen")
            model_pred[int(i * len_run) : int((i + 1) * len_run)] = model_pred_temp

        da = xr.DataArray(
            data=model_pred,
            dims=["time", "x", "y", "var"],
        )

        da.to_zarr(
            pred_model_path
            + "/Pred_Short_Data_025_"
            + region
            + "_in_"
            + str_in
            + "ext_"
            + str_ext
            + "N_samples_"
            + str(N_samples)
            + "_rand_seed_"
            + str(rand_ind)
            + ".zarr",
            mode="w",
        )

4000 1
data_gen
4000 1
data_gen
4000 1
data_gen
4000 1
data_gen
4000 1
data_gen
4000 2
data_gen
4000 2
data_gen
4000 2
data_gen
4000 2
data_gen
4000 2
data_gen
4000 3
data_gen
4000 3
data_gen
4000 3
data_gen
4000 3
data_gen
4000 3
data_gen
